## Model Evaluation

### Purpose
This notebook evaluates a registered model version before promotion.

It loads the current candidate model, runs inference on evaluation data, and logs performance metrics using MLflow’s evaluation framework.

This is the first step in Stage 3 of the BYOM workflow (Evaluate → Approve → Deploy).

### What This Notebook Does

- Loads the registered model from Unity Catalog
- Runs predictions on evaluation data
- Computes performance metrics (e.g., accuracy, F1, etc.)
- Logs evaluation results to MLflow
- Tags the evaluated model version
- Sets task values required for downstream approval logic

This notebook does **not** promote or deploy the model.

In [0]:
%pip install -q threadpoolctl>=3.5.0 xgboost torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [0]:
# Widget parameters for job orchestration
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("model_name", "pyfunc_xgb", "Model Name")  # same as model_type when logging, or dl model name from artifacts.json
dbutils.widgets.text("model_version", "", "Model Version (leave empty for latest)")
dbutils.widgets.text("evaluation_table", "", "Evaluation Data Table (leave empty to use features_table_name)")
dbutils.widgets.text("features_table_name", "iris_features", "Features Table Name")
dbutils.widgets.text("labels_table_name", "iris_labels", "Labels Table Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
features_table_name = dbutils.widgets.get("features_table_name")
labels_table_name = dbutils.widgets.get("labels_table_name")
evaluation_table = dbutils.widgets.get("evaluation_table") or f"{catalog}.{schema}.{features_table_name}"
labels_table = f"{catalog}.{schema}.{labels_table_name}"

registered_model_name = f"{catalog}.{schema}.{model_name}"

### Load Registered Model

Loads the model using the configured:

- `catalog_name`
- `schema_name`
- `model_name`

The notebook references the appropriate model version (typically the latest candidate or specific version ID).


In [ ]:
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Resolve model version
if not model_version:
    versions = client.search_model_versions(f"name='{registered_model_name}'")
    if not versions:
        raise ValueError(f"No versions found for model: {registered_model_name}")
    model_version = max(v.version for v in versions)

# Load model and evaluation data
model_uri = f"models:/{registered_model_name}/{model_version}"
model = mlflow.pyfunc.load_model(model_uri)
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]

eval_df = spark.table(evaluation_table)
labels_df = spark.table(labels_table)
eval_pdf = eval_df.join(labels_df, on="id", how="inner").toPandas()
X_eval = eval_pdf[feature_cols]
y_true = eval_pdf["target"]

### DL PyFunc Model Class (Required for 1d Only)

If evaluating a PyFunc deep learning model (from `1d_dl_pyfunc.ipynb`),  
the model class must be redefined here to ensure correct deserialization.

Skip this section for native framework models.

In [ ]:
import torch
import torch.nn as nn

# Must match the class used in 0_export and 1d_dl_pyfunc when the DL PyFunc was logged.
class SimpleClassifierNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim), nn.ReLU(),
            nn.Linear(hid_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

### Run Evaluation Dataset Scoring

Uses evaluation data to:
- Generate predictions
- Compare against ground truth labels
- Prepare results for MLflow evaluation

Ensure evaluation data reflects realistic production conditions.

In [ ]:
# Predict and build eval dataframe (handle DataFrame, Series, or ndarray; column may be "prediction" or first column)
y_pred_out = model.predict(X_eval)
if hasattr(y_pred_out, "columns") and "prediction" in y_pred_out.columns:
    y_pred = y_pred_out["prediction"].values
elif hasattr(y_pred_out, "columns"):
    y_pred = y_pred_out.iloc[:, 0].values
elif hasattr(y_pred_out, "values"):
    y_pred = np.asarray(y_pred_out).ravel()
else:
    y_pred = np.asarray(y_pred_out).ravel()

### MLflow Evaluation & Metrics Logging

Runs `mlflow.evaluate()` to:

- Log performance metrics
- Capture evaluation artifacts
- Enable reproducible comparison across model versions

Metrics logged here are used by the approval stage.

In [ ]:
# Classifier metrics require integer class labels; native PyTorch (and some wrappers) may return logits/probs
y_pred = np.asarray(y_pred)
if y_pred.ndim == 2:
    y_pred = np.argmax(y_pred, axis=1)
elif np.issubdtype(y_pred.dtype, np.floating):
    y_pred = np.round(y_pred).astype(np.int64)
eval_data = X_eval.copy()
eval_data["target"] = y_true.values
eval_data["prediction"] = y_pred

# Run MLflow evaluation (mlflow.models.evaluate = non-deprecated API)
with mlflow.start_run(run_name=f"evaluation_{model_name}_v{model_version}") as eval_run:
    results = mlflow.models.evaluate(
        data=eval_data,
        targets="target",
        predictions="prediction",
        model_type="classifier",
        evaluators=["default"],
    )
    mlflow.log_param("model_name", registered_model_name)
    mlflow.log_param("model_version", model_version)
    mlflow.log_param("evaluation_table", evaluation_table)
    mlflow.log_param("num_samples", len(eval_data))
    eval_run_id = eval_run.info.run_id

# Metrics for approval (mlflow.models.evaluate uses accuracy_score, f1_score)
metrics = results.metrics
accuracy = metrics.get("accuracy_score", 0)
f1_weighted = metrics.get("f1_score", 0)

# Tag model version and set task values for downstream jobs
client.set_model_version_tag(registered_model_name, model_version, "evaluation_status", "completed")
client.set_model_version_tag(registered_model_name, model_version, "evaluation_run_id", eval_run_id)
client.set_model_version_tag(registered_model_name, model_version, "accuracy", str(round(accuracy, 4)))
client.set_model_version_tag(registered_model_name, model_version, "f1_weighted", str(round(f1_weighted, 4)))

dbutils.jobs.taskValues.set(key="model_name", value=registered_model_name)
dbutils.jobs.taskValues.set(key="model_version", value=str(model_version))
dbutils.jobs.taskValues.set(key="accuracy", value=str(accuracy))
dbutils.jobs.taskValues.set(key="f1_weighted", value=str(f1_weighted))
dbutils.jobs.taskValues.set(key="evaluation_run_id", value=eval_run_id)

print(f"Evaluation completed for {registered_model_name} version {model_version}")
print(f"Accuracy: {accuracy:.4f}  F1 (weighted): {f1_weighted:.4f}")
